# Retrieval quality — evaluare cuantificată

Rulează cele 10 întrebări din `tests.jsonl` prin pipeline-ul de retrieval (`retrieve_with_trace`), calculează MRR / nDCG / recall / precision / hit-rate pe fiecare din cele 4 etape (semantic, lexical, fused, reranked), și le compară.

Presupune că datele sunt deja ingerate în Postgres (vezi `README.md`). Pentru inspecție liberă, nemetrizată, pe întrebări ad-hoc, vezi [`retrieval_playground.ipynb`](../retrieval/retrieval_playground.ipynb).

## Known limitations

- **`reranked` here is `trace.reranked`, not `trace.final`.** `RetrievalTrace.final` is `reranked[:FINAL_K]` — the list the real answer pipeline actually consumes. Today `FINAL_K == RETRIEVAL_K == 20`, so the two lists are identical and every metric below is correct as computed. But this means a `FINAL_K` change alone will **not** show up in this notebook's numbers, since they're computed over the untruncated `reranked` list. If you're specifically evaluating a `FINAL_K` change, add a 5th `evaluate_stage("final", trace.final, test.keywords)` stage before trusting a snapshot diff to reflect it.
- **Keyword matching is over `doc.page_content`.** Every metric here (via `keyword_hit_ranks`, and the reused `calculate_mrr`/`calculate_ndcg` from `eval.py`) checks whether a keyword substring appears in `page_content`. If `ingestion/contextualize.py`'s `add_context` is re-enabled (currently commented out in `ingest.py`), `page_content` becomes LLM-generated context prepended to the real chunk text — a keyword can then match text the LLM wrote *about* the chunk, inflating recall/precision/MRR/nDCG for reasons unrelated to retrieval quality. **A `"contextualize-on"` snapshot is not apples-to-apples with a `"contextualize-off"` one** — don't diff them and conclude contextualize helped without checking by hand. (The breakdown cell below previews `metadata.get("original_text", page_content)`, so once contextualize is on, a "found at [stage]" line can point at text that doesn't itself contain the keyword — the actual match happened in the prepended LLM context.)

In [1]:
from datetime import UTC, datetime

from dotenv import load_dotenv

load_dotenv(override=True)

from app.rag_simple import config
from app.rag_simple.evaluation.retrieval_metrics import (
    average_stage_metrics,
    evaluate_trace,
    keyword_hit_ranks,
)
from app.rag_simple.evaluation.results import EvalSnapshot, diff_snapshots, list_snapshots
from app.rag_simple.evaluation.test import load_tests
from app.rag_simple.retrieval.pipeline import retrieve_with_trace

RESULTS_DIR = config.ROOT / "evaluation" / "results"
STAGES = ("semantic", "lexical", "fused", "reranked")
METRIC_COLUMNS = (
    "mrr",
    "ndcg",
    "recall_at_5",
    "recall_at_10",
    "precision_at_5",
    "precision_at_10",
    "hit_rate_at_10",
)

print(f"RETRIEVAL_K={config.RETRIEVAL_K}  FINAL_K={config.FINAL_K}  CHAT_MODEL={config.CHAT_MODEL}")

/home/alex/projects/RAG/full-stack-fastapi-template/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


RETRIEVAL_K=20  FINAL_K=20  CHAT_MODEL=openai:gpt-4.1-mini


In [16]:
tests = load_tests()
traces = []
per_question_metrics = []

for test in tests:
    trace = retrieve_with_trace(test.question)
    traces.append(trace)
    per_question_metrics.append(evaluate_trace(trace, test))

print(f"Evaluated {len(tests)} questions across stages {STAGES}")

Evaluated 39 questions across stages ('semantic', 'lexical', 'fused', 'reranked')


In [17]:
aggregated = average_stage_metrics(per_question_metrics)

header = f"{'stage':<10}" + "".join(f"{col:>16}" for col in METRIC_COLUMNS)
print(header)
print("-" * len(header))
for stage in STAGES:
    row = f"{stage:<10}" + "".join(f"{aggregated[stage][col]:>16.3f}" for col in METRIC_COLUMNS)
    print(row)

stage                  mrr            ndcg     recall_at_5    recall_at_10  precision_at_5 precision_at_10  hit_rate_at_10
--------------------------------------------------------------------------------------------------------------------------
semantic             0.627           0.678           0.885           0.911           0.249           0.156           0.949
lexical              0.397           0.473           0.621           0.721           0.160           0.107           0.769
fused                0.620           0.666           0.800           0.903           0.219           0.144           0.923
reranked             0.915           0.888           0.925           0.925           0.273           0.159           0.949


## Cum citești tabelul de mai sus

**Rândurile = etapele pipeline-ului** (`retrieval/pipeline.py`), fiecare producând o listă ordonată de max. 20 de chunk-uri:

| etapă | ce face |
|---|---|
| `semantic` | căutare vectorială (pgvector) pe întrebarea originală + pe cea rescrisă de LLM, cele 2 liste unite prin RRF |
| `lexical` | BM25 (potrivire de cuvinte exacte, fără stemming) pe aceleași 2 variante ale întrebării, unite prin RRF |
| `fused` | `semantic` + `lexical` unite prin Reciprocal Rank Fusion |
| `reranked` | lista `fused` reordonată de un LLM (`CHAT_MODEL`) — **asta e lista pe care o primește generarea** |

**Coloanele = metrici.** Toate se bazează pe aceeași definiție de „relevant": **un chunk e relevant pentru un keyword dacă textul lui conține keyword-ul ca substring** (case-insensitive). Fiecare întrebare din `tests.jsonl` are o listă de `keywords`; metricile se calculează per keyword, se mediază pe keyword-urile întrebării, apoi pe toate întrebările.

Consecință importantă: potrivirea e **literală**. `'caz terminal'` nu se potrivește cu „Cazul terminal", `'aroganță'` nu se potrivește cu „Aroganța". Nu există stemming sau toleranță la forme flexionare.

### `mrr` — Mean Reciprocal Rank

**Întrebarea la care răspunde:** cât de sus apare *primul* chunk care conține keyword-ul?

Per keyword: `1 / rangul primului chunk care îl conține` (căutat în toată lista de 20), sau `0` dacă nu apare deloc.
- pe locul 1 → 1.0 · pe locul 2 → 0.5 · pe locul 3 → 0.33 · pe locul 10 → 0.1 · absent → 0

Apoi media pe keyword-uri → media pe întrebări.

**Interpretare:** 1.0 = fiecare keyword e în chunk-ul de pe primul loc. E metrica cea mai sensibilă la ordine — penalizează dur coborârea de pe locul 1 pe 2. `reranked = 0.900` vs `lexical = 0.581` înseamnă că, după rerank, informația căutată e aproape mereu în primul chunk, pe când BM25 singur o pune în medie mai jos.

### `ndcg` — normalized Discounted Cumulative Gain (@10)

**Întrebarea la care răspunde:** chunk-urile care conțin keyword-ul sunt grupate *în vârful* top-10, sau împrăștiate spre coadă?

Per keyword, pe primele 10 chunk-uri: fiecare chunk relevant aduce `1 / log2(rang + 1)` (locul 1 → 1.0, locul 2 → 0.63, locul 3 → 0.5, locul 10 → 0.29). Suma (DCG) se împarte la suma ideală (IDCG) — aceleași chunk-uri relevante, dar puse pe primele poziții.

Diferență față de MRR: MRR se uită doar la **primul** hit; nDCG le numără pe **toate** hit-urile din top 10 și cât de sus sunt.

**Atenție la implementarea de aici** (`eval.py::calculate_ndcg`): idealul se calculează doar din chunk-urile *găsite* în top 10, nu din toate chunk-urile relevante din corpus. Deci nDCG măsoară **ordinea** din top 10, nu cât din corpus a fost găsit. Dacă keyword-ul nu apare deloc în top 10 → 0.

### `recall_at_5` și `recall_at_10`

**Întrebarea la care răspunde:** ce fracțiune din keyword-urile așteptate apare *undeva* în primele k chunk-uri?

Per întrebare: `nr. keyword-uri găsite în top k / nr. total keyword-uri`. Ordinea în interiorul top k nu contează.

Exemplu: 5 keyword-uri, 4 găsite în top 5 → `recall_at_5 = 0.8`.

**Interpretare:** e metrica „a ajuns informația în context?". Pentru RAG e cea mai importantă: dacă un fapt nu e în chunk-urile trimise la LLM, răspunsul nu-l poate conține. `recall_at_10` ≥ `recall_at_5` mereu.

În rularea curentă `recall_at_10 = 0.920` la toate etapele; tot deficitul de 0.08 vine dintr-o singură întrebare (vezi mai jos, „Cum arată rezultatele").

### `precision_at_5` și `precision_at_10`

**Întrebarea la care răspunde:** cât din top k e „ocupat" de chunk-uri care conțin keyword-urile?

Per keyword: `nr. chunk-uri din top k care conțin keyword-ul / k`. Apoi media pe keyword-uri, apoi pe întrebări.

**Atenție: nu e precizia clasică și nu poate ajunge la 1.0 în practică.** Dacă o întrebare are 5 keyword-uri și fiecare apare în exact 1 chunk din top 5 (retrieval perfect), precision_at_5 = 1/5 = **0.2**. Valoarea mare apare doar când *același* keyword se repetă în multe chunk-uri — ceea ce se întâmplă mai ales cu keyword-uri generice (`legume`, `foc`, `cap`).

**Interpretare:** folosește-o doar **relativ** (etapă vs etapă, snapshot vs snapshot), nu ca procent absolut. `0.348` la `reranked` vs `0.289` la `semantic` spune că reranker-ul a adus mai multe chunk-uri „cu vorbele potrivite" în top 5 — nu că 65% din context e zgomot.

### `hit_rate_at_10`

**Întrebarea la care răspunde:** a găsit etapa *măcar ceva* util?

Per întrebare, binar: `1` dacă **cel puțin un** keyword apare în top 10, altfel `0`. Media pe întrebări = procentul de întrebări pentru care retrieval-ul n-a eșuat complet.

**Interpretare:** e un detector de eșecuri totale. `1.000` peste tot = nicio întrebare n-a rămas fără niciun chunk relevant. Pentru că e binar și pragul e jos (un singur keyword, adesea generic), e **saturat** pe setul actual și nu mai diferențiază etapele.

## Breakdown per întrebare

Tabelul de sus e o medie; aici vezi **ce keyword-uri anume** strică media. Celula de mai jos parcurge fiecare pereche (întrebare, keyword) și afișează doar perechile „interesante":

- **ratat la final** — keyword-ul nu apare în lista `reranked` (nici un chunk nu-l conține), sau
- **salt de rang** — keyword-ul e găsit, dar rangul lui diferă cu mai mult de `RANK_SHIFT_THRESHOLD` poziții între etape (ex. locul 1 la `semantic`, locul 9 la `lexical`) — arată care etapă a ajutat/încurcat.

Keyword-urile care sunt găsite și stabile la toate etapele **nu** apar. O întrebare apare o singură dată, cu toate keyword-urile ei interesante dedesubt.

Cum citești o linie:
- `ranks: {'semantic': 1, 'lexical': 9, ...}` = pe ce poziție (1–20) e primul chunk care conține keyword-ul, la fiecare etapă; `None` = nu apare deloc în cele 20.
- `găsit la [etapă]` = chunk-ul respectiv (sursă | heading | început de text), luat de la cea mai avansată etapă unde apare — ca să verifici cu ochiul dacă e chiar chunk-ul corect.


In [20]:
RANK_SHIFT_THRESHOLD = 5  # out of RETRIEVAL_K=20, a shift bigger than this is a meaningful stage effect


def show_chunk(doc):
    source = doc.metadata.get("source", "?")
    heading = doc.metadata.get("h4") or doc.metadata.get("h3") or ""
    text = doc.metadata.get("original_text", doc.page_content)
    preview = text.strip().replace("\n", " ")
    if len(preview) > 220:
        preview = preview[:220] + "..."
    return f"{source} | {heading}\n         {preview}"


for test, trace in zip(tests, traces, strict=True):
    stage_docs = {
        "semantic": trace.semantic_hits,
        "lexical": trace.lexical_hits,
        "fused": trace.fused,
        "reranked": trace.reranked,
    }
    flagged = []
    for keyword in test.keywords:
        ranks = {stage: keyword_hit_ranks(keyword, docs) for stage, docs in stage_docs.items()}
        found_ranks = [rank for rank in ranks.values() if rank is not None]
        missed_final = ranks["reranked"] is None
        shifted = bool(found_ranks) and (max(found_ranks) - min(found_ranks) > RANK_SHIFT_THRESHOLD)
        if missed_final or shifted:
            flagged.append((keyword, ranks, "ratat la final" if missed_final else "salt de rang"))

    if not flagged:
        continue

    print(f"Q: {test.question}")
    print(f"   {len(flagged)}/{len(test.keywords)} keyword-uri semnalate")
    for keyword, ranks, reason in flagged:
        print(f"   - {keyword!r} [{reason}]  ranks: {ranks}")
        best_stage = next(
            (stage for stage in ("reranked", "fused", "lexical", "semantic") if ranks[stage] is not None),
            None,
        )
        if best_stage is not None:
            doc = stage_docs[best_stage][ranks[best_stage] - 1]
            print(f"     găsit la [{best_stage}]: {show_chunk(doc)}")
        else:
            print("     negăsit la nicio etapă")
    print()

Q: Ce organe corespund fiecărui tip de cereale, conform corespondenței prezentate în seminar?
   1/8 keyword-uri semnalate
   - 'hrișcă' [salt de rang]  ranks: {'semantic': 4, 'lexical': 1, 'fused': 1, 'reranked': 12}
     găsit la [reranked]: seminar macrobiotica.md | Dulciurile Naturale și Anatomia Evoluției
         #### Dulciurile Naturale și Anatomia Evoluției   Avem sucul de orez și *sufleul* de hrișcă. Dacă doriți să aveți o mâncare bună, utilizați un pic de orez. Printre micile plăceri, dulceața naturală din cereale este cea ma...

Q: Care sunt cele 5 dificultăți majore în vindecarea cancerului prin macrobiotică, conform Michio Kushi?
   5/5 keyword-uri semnalate
   - 'inexactități în dietă' [salt de rang]  ranks: {'semantic': 2, 'lexical': 10, 'fused': 4, 'reranked': 2}
     găsit la [reranked]: seminar macrobiotica.md | Studii de Caz și Statistici Internaționale
         Cauzele eșecului în macrobiotică sunt: 1) Inexactități în dietă; 2) Practicarea a mai multor metode în ace

### Cum arată rezultatele rulării cu 10 întrebări (baseline, 2026-09-28)

**1. Majoritatea „eșecurilor" din breakdown sunt eșecuri ale testului, nu ale retrieval-ului.** La întrebarea Kushi, chunk-ul corect e pe locul 2 după rerank și conține textual toate cele 5 cauze: *„1) Inexactități în dietă; 2) Practicarea a mai multor metode…; 3) Cazul terminal…; 4) Lipsa cooperării din partea familiei; 5) Aroganța."* Dar keyword-urile din `tests.jsonl` sunt scrise în altă formă (`'mai multe metode'`, `'caz terminal'`, `'lipsa cooperării familiei'`, `'aroganță'`), iar substring-ul nu se potrivește. Remediu: keyword-uri care apar **literal** în sursă (ex. `'mai multor metode'`, `'cazul terminal'`, `'cooperării'`, `'aroganța'`).

**2. Acest artefact explică aproape tot deficitul din tabel.** Întrebarea Kushi are recall 1/5 = 0.2 → trage media pe 10 întrebări cu 0.8/10 = **0.08**, exact diferența dintre `recall_at_10 = 0.920` și 1.0. Deci celelalte 9 întrebări au recall@10 = 100% la toate etapele. La fel, `reranked mrr = 0.900` ≈ maximul posibil cu acest artefact (~0.91) → după rerank, informația e aproape mereu pe locul 1.

**3. Setul de test e prea ușor ca să diferențieze variante de chunking/ingestie.** `hit_rate` și `recall_at_10` sunt saturate. O schimbare de chunking care strică lucrurile nu s-ar vedea în ele; s-ar vedea cel mult în `mrr`/`ndcg`/`recall_at_5`.

**4. Ce se poate totuși concluziona:**
- `lexical` (BM25) e etapa cea mai slabă (MRR 0.58, recall@5 0.74) — firesc pentru română fără stemming: „cancerului" ≠ „cancer". Contribuie mai ales ca plasă de siguranță în `fused`.
- `reranked` îmbunătățește ordinea (MRR 0.86 → 0.90, precision@5 0.29 → 0.35), dar coboară ușor un keyword din top 5 (recall@5 0.920 → 0.907).
- `fused` e *mai slab* decât `semantic` singur la MRR (0.78 vs 0.86): RRF-ul trage în jos rezultatele bune semantic cu cele slabe lexical; rerank-ul repară asta.

**5. Zgomot statistic.** Cu 10 întrebări, o singură întrebare = 10 puncte procentuale la `hit_rate` și ~0.1 la MRR. Reranker-ul e un LLM (nedeterminist chiar la temperature 0), deci diferențe de ~0.02–0.04 între snapshot-uri nu sunt semnificative. Mai multe întrebări (acum `tests.jsonl` are 39) reduc zgomotul.

**6. Limita de fond a metodei:** „relevant = conține substring-ul" nu verifică dacă chunk-ul chiar răspunde la întrebare. Keyword-urile scurte/generice (`cap` se potrivește și în „capitol", `foc` în „focus", `legume` apare în zeci de chunk-uri) umflă scorurile. Pentru o evaluare de ingestie reală, pasul următor ar fi relevanța la nivel de chunk/fragment sursă (ex. „chunk-ul corect e cel care conține pasajul X") sau un LLM-judge pe context.

## Salvare snapshot

Editează `LABEL` mai jos înainte de a rula celula (ex. `"baseline"`, `"contextualize-on"`).

In [11]:
LABEL = "baseline"

snapshot = EvalSnapshot(
    label=LABEL,
    timestamp=datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ"),
    config={
        "RETRIEVAL_K": config.RETRIEVAL_K,
        "FINAL_K": config.FINAL_K,
        "CHAT_MODEL": config.CHAT_MODEL,
        "EMBEDDING_MODEL": config.EMBEDDING_MODEL,
    },
    stage_metrics=aggregated,
)
saved_path = snapshot.save(RESULTS_DIR)
print(f"Saved snapshot: {saved_path}")

Saved snapshot: /home/alex/projects/RAG/full-stack-fastapi-template/backend/app/rag_simple/evaluation/results/20260928T150706Z_baseline.json


## Comparație before/after

Încarcă ultimele două snapshot-uri salvate și afișează delta per etapă/metrică.

In [12]:
snapshots = list_snapshots(RESULTS_DIR)

if len(snapshots) >= 2:
    older, newer = snapshots[-2], snapshots[-1]
    delta = diff_snapshots(older, newer)
    print(f"Comparing {older.label!r} ({older.timestamp}) -> {newer.label!r} ({newer.timestamp})")
    for stage in STAGES:
        print(f"\n[{stage}]")
        for metric in METRIC_COLUMNS:
            value = delta[stage][metric]
            sign = "+" if value >= 0 else ""
            print(f"  {metric:<16} {sign}{value:.3f}")
else:
    print("Need at least two saved snapshots to compare — run this notebook again after a change.")

Comparing 'baseline' (20260928T150547Z) -> 'baseline' (20260928T150706Z)

[semantic]
  mrr              +0.000
  ndcg             +0.000
  recall_at_5      +0.000
  recall_at_10     +0.000
  precision_at_5   +0.000
  precision_at_10  +0.000
  hit_rate_at_10   +0.000

[lexical]
  mrr              +0.000
  ndcg             +0.000
  recall_at_5      +0.000
  recall_at_10     +0.000
  precision_at_5   +0.000
  precision_at_10  +0.000
  hit_rate_at_10   +0.000

[fused]
  mrr              +0.000
  ndcg             +0.000
  recall_at_5      +0.000
  recall_at_10     +0.000
  precision_at_5   +0.000
  precision_at_10  +0.000
  hit_rate_at_10   +0.000

[reranked]
  mrr              +0.000
  ndcg             +0.000
  recall_at_5      +0.000
  recall_at_10     +0.000
  precision_at_5   +0.000
  precision_at_10  +0.000
  hit_rate_at_10   +0.000
